# Check cached boxes for teddy bear and bed
Attach **objectdetectionecache** and **upt-prompts-full-all-actions-v2**. Select CPU. Send the resulting `cider_error_two_source_boxes.json`.

In [ ]:
from pathlib import Path
import hashlib
import json
import math

INPUT = Path('/kaggle/input/datasets/ducanh2403')
SOURCES = {
    'upt': (INPUT/'upt-prompts-full-all-actions-v2/prompt_cache.json',
            '24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d'),
    'detector': (INPUT/'objectdetectionecache/objectdetectioncache.json',
                 '498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c'),
}
TARGETS = {239894: ('teddy bear', 426, 640), 487438: ('bed', 640, 480)}
OUTPUT = Path('/kaggle/working/cider_error_two_source_boxes.json')

def checked_json(path, expected):
    print('Checking', path.name, flush=True)
    with path.open('rb') as stream:
        if hashlib.file_digest(stream, 'sha256').hexdigest() != expected:
            raise ValueError(f'Unexpected input bytes: {path}')
        stream.seek(0)
        return json.load(stream)

def normalized_box(values, width, height):
    limits = (width, height, width, height)
    box = [min(max(float(value), 0), limit) / limit
           for value, limit in zip(values, limits, strict=True)]
    if not (box[0] < box[2] and box[1] < box[3]):
        raise ValueError('Invalid clipped ROI')
    return box

def iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    intersection = iw * ih
    return intersection / ((a[2]-a[0])*(a[3]-a[1]) +
                           (b[2]-b[0])*(b[3]-b[1]) - intersection)

def label(text):
    return ' '.join(text.lower().replace('_', ' ').split())

def trace(image_id, target, width, height, detected, upt, linked):
    if isinstance(detected, dict):
        detected = detected['objects']
    ordered = sorted(enumerate(detected), key=lambda x: (-float(x[1]['conf']), x[0]))[:150]
    rois = [dict(roi=i, label=label(obj['label']), confidence=float(obj['conf']),
                 bbox=normalized_box(obj['bbox'], width, height))
            for i, (_, obj) in enumerate(ordered)]
    if len(rois) != linked['roi_count']:
        raise ValueError(f'Reconstructed ROI count differs for {image_id}')
    for obj in linked['objects']:
        roi = rois[obj['roi']]
        if roi['label'] != obj['label'] or not all(
            math.isclose(a, b, abs_tol=1e-6, rel_tol=0)
            for a, b in zip(roi['bbox'], obj['roi_bbox'], strict=True)
        ):
            raise ValueError(f'Reconstructed box differs for {image_id} ROI {roi["roi"]}')
    target_rois = [r for r in rois if r['label'] == target]
    target_upt = [o for o in upt['objects'] if label(o['label']) == target]
    overlaps = [dict(upt_index=o['index'], roi=r['roi'], iou=iou(o['bbox'], r['bbox']))
                for o in target_upt for r in target_rois]
    return dict(coco_id=image_id, target=target, width=width, height=height, detector_rois=rois,
                upt_mode=upt['mode'], upt_objects=upt['objects'], upt_relations=upt['relations'],
                target_rois=target_rois, target_upt_objects=target_upt, target_pair_ious=overlaps,
                linked_prompt=linked)

def main():
    links = LINKED_ROWS
    wanted = {row['filename']: row for row in links.values()}
    detector = checked_json(*SOURCES['detector'])
    if 'data' in detector:
        detector = detector['data']
    selected = {}
    for path, record in detector.items():
        name = Path(path).name
        if name in wanted:
            if name in selected:
                raise ValueError(f'Duplicate detector filename {name}')
            selected[name] = record
    del detector
    upt = checked_json(*SOURCES['upt'])
    rows = []
    for image_id, config in TARGETS.items():
        linked = links[image_id]
        name = linked['filename']
        rows.append(trace(image_id, *config, selected[name], upt[name], linked))
    result = dict(sources={name: expected for name, (_, expected) in SOURCES.items()}, rows=rows)
    OUTPUT.write_text(json.dumps(result, ensure_ascii=False, allow_nan=False, indent=2)+'\n', encoding='utf-8')
    print(OUTPUT)

LINKED_ROWS = {239894: {'coco_id': 239894, 'filename': 'COCO_val2014_000000239894.jpg', 'group': 'val', 'roi_count': 11, 'source_mode': 'object_only', 'objects': [{'instance_id': 'det_0270b3a10178', 'upt_index': 0, 'roi': 0, 'label': 'person', 'bbox': [0.2610907554626465, 0.11613607922759679, 0.7919361877441407, 0.9934927680708784], 'roi_bbox': [0.27180751388621444, 0.12229686975479126, 0.7944600996277142, 0.9990468978881836], 'IoU': 0.9623897040803431, 'confidence': 0.995604395866394, 'text': 'a person'}], 'relations': [], 'counts': {'objects': 1, 'relations': 0, 'mapped_objects': 1, 'kept_objects': 1, 'mapped_relations': 0, 'capped_objects': 0}}, 487438: {'coco_id': 487438, 'filename': 'COCO_val2014_000000487438.jpg', 'group': 'val', 'roi_count': 14, 'source_mode': 'object_only', 'objects': [{'instance_id': 'det_ae24eaa745e8', 'upt_index': 1, 'roi': 0, 'label': 'suitcase', 'bbox': [0.19244177211740005, 0.80328369140625, 0.5194768565680699, 0.9949711608886719], 'roi_bbox': [0.19817187786102294, 0.8068958282470703, 0.5137499809265137, 0.9912708282470704], 'IoU': 0.9281556741271637, 'confidence': 0.9869580864906311, 'text': 'a suitcase'}, {'instance_id': 'det_261e0a8e7ee9', 'upt_index': 2, 'roi': 6, 'label': 'backpack', 'bbox': [0.37074377120771285, 0.5315413665771485, 0.557257217493111, 0.7438865661621094], 'roi_bbox': [0.3778125047683716, 0.5420208613077799, 0.5994531154632569, 0.7419375101725261], 'IoU': 0.7467384560899055, 'confidence': 0.8067554235458374, 'text': 'a backpack'}], 'relations': [], 'counts': {'objects': 3, 'relations': 0, 'unmatched_objects': 1, 'mapped_objects': 2, 'kept_objects': 2, 'mapped_relations': 0, 'capped_objects': 0}}}
main()
